# 2. lépés: beágyazás és index

**Bemenet:** `chunks/en.jsonl`, `chunks/hu.jsonl`
**Kimenet:** `index/en.npy`, `index/hu.npy`, `index/info.json`

In [1]:
%pip install -q pandas numpy requests tqdm

Note: you may need to restart the kernel to use updated packages.


## Beállítások

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(r"C:\path\to\project")

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Folder not found: {PROJECT_DIR} - set PROJECT_DIR to your project folder.")
os.chdir(PROJECT_DIR)
print("Working folder:", Path.cwd())

CHUNKS_DIR = Path("chunks")
INDEX_DIR = Path("index")
LANGS = ["en", "hu"]

EMBED_URL = "http://192.168.88.10:1234/v1"
EMBED_MODEL = "text-embedding-multilingual-e5-large-instruct"

FORMATS = {
    "text-embedding-multilingual-e5-large-instruct": {
        "query": "Instruct: Given a question about a novel, retrieve passages from the novel that answer the question\nQuery: {text}",
        "passage": "{text}",
    },
    "text-embedding-bge-m3": {"query": "{text}", "passage": "{text}"},
}
BATCH_SIZE = 1

INDEX_DIR.mkdir(exist_ok=True)

In [ ]:
import requests, time, numpy as np

def embed(texts, model, url, api_key="lm-studio", batch=32, retries=3):
    """Embed texts via the LM Studio /v1/embeddings endpoint. Returns an L2-normalized float32 matrix."""
    vectors = []
    for i in range(0, len(texts), batch):
        part = texts[i:i + batch]
        for attempt in range(retries):
            try:
                r = requests.post(f"{url}/embeddings",
                                  headers={"Authorization": f"Bearer {api_key}"},
                                  json={"model": model, "input": part}, timeout=600)
                r.raise_for_status()
                data = sorted(r.json()["data"], key=lambda d: d["index"])
                vectors += [d["embedding"] for d in data]
                break
            except Exception as e:
                if attempt == retries - 1:
                    raise
                print(f"   error ({e}), retrying...")
                time.sleep(5 * (attempt + 1))
    X = np.asarray(vectors, dtype="float32")
    X /= np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    return X

r = requests.get(f"{EMBED_URL}/models", timeout=30)
r.raise_for_status()
available = [m["id"] for m in r.json().get("data", [])]
print("Models on the embedding server:", available)
print("Configured model:", EMBED_MODEL, "->", "OK" if EMBED_MODEL in available else "NOT IN THE LIST")

Models on the embedding server: ['text-embedding-bge-m3', 'text-embedding-multilingual-e5-large-instruct', 'text-embedding-qwen3-embedding-4b', 'google/gemma-4-e4b', 'text-embedding-nomic-embed-text-v1.5']
Configured model: text-embedding-multilingual-e5-large-instruct -> OK


## A darabok beolvasása és beágyazása

In [ ]:
import hashlib

def fingerprint(texts):
    """Hash of all chunk texts - detects re-chunking without re-indexing."""
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8")); h.update(b"\0")
    return h.hexdigest()

import pandas as pd, json, datetime
from tqdm.auto import tqdm

info = {"model": EMBED_MODEL, "url": EMBED_URL, "format": FORMATS[EMBED_MODEL], "batch_size": BATCH_SIZE,
        "date": datetime.datetime.now().isoformat(timespec="seconds"), "langs": {}}

old_info = {}
if (INDEX_DIR / "info.json").exists():
    old_info = json.loads((INDEX_DIR / "info.json").read_text(encoding="utf-8"))

for lang in LANGS:
    df = pd.read_json(CHUNKS_DIR / f"{lang}.jsonl", lines=True)
    fp = fingerprint(df["text"])
    prev = old_info.get("langs", {}).get(lang, {})
    if (prev.get("fingerprint") == fp and old_info.get("model") == EMBED_MODEL
            and old_info.get("format") == FORMATS[EMBED_MODEL] and old_info.get("batch_size") == BATCH_SIZE
            and (INDEX_DIR / f"{lang}.npy").exists()):
        info["langs"][lang] = prev
        print(f"[{lang}] unchanged since last indexing - kept existing index/{lang}.npy")
        continue
    
    part_file = INDEX_DIR / f"{lang}.partial.npy"
    part_meta = INDEX_DIR / f"{lang}.partial.json"
    texts = [FORMATS[EMBED_MODEL]["passage"].format(text=t) for t in df["text"]]
    done = []
    if part_file.exists() and part_meta.exists():
        meta = json.loads(part_meta.read_text(encoding="utf-8"))
        saved = np.load(part_file)
        if (meta.get("fingerprint") == fp and meta.get("model") == EMBED_MODEL
                and meta.get("batch_size") == BATCH_SIZE and len(saved) <= len(texts)):
            done = list(saved)
            print(f"[{lang}] resuming: {len(done):,} of {len(texts):,} chunks already embedded")
    print(f"[{lang}] embedding {len(texts) - len(done):,} chunks ...")
    t0 = time.time()
    STEP = max(BATCH_SIZE * 10, 200)
    for start in tqdm(range(len(done), len(texts), STEP), desc=f"{lang}", unit="step"):
        done += list(embed(texts[start:start + STEP], EMBED_MODEL, EMBED_URL, batch=BATCH_SIZE))
        np.save(part_file, np.asarray(done, dtype="float32"))
        part_meta.write_text(json.dumps({"fingerprint": fp, "model": EMBED_MODEL, "batch_size": BATCH_SIZE}), encoding="utf-8")
    X = np.asarray(done, dtype="float32")
    np.save(INDEX_DIR / f"{lang}.npy", X)
    part_file.unlink(missing_ok=True); part_meta.unlink(missing_ok=True)
    info["langs"][lang] = {"n_chunks": len(df), "dim": int(X.shape[1]),
                           "fingerprint": fp, "seconds": round(time.time() - t0, 1)}
    print(f"     done: {X.shape[0]} x {X.shape[1]} matrix, {time.time() - t0:.0f} s")

_ = (INDEX_DIR / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved to:", INDEX_DIR.resolve())

C:\Users\<user>\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[en] embedding 3,592 chunks ...


en: 100%|██████████| 18/18 [1:22:25<00:00, 274.73s/step]


     done: 3592 x 1024 matrix, 4945 s
[hu] embedding 2,919 chunks ...


hu: 100%|██████████| 15/15 [1:29:01<00:00, 356.09s/step]

     done: 2919 x 1024 matrix, 5341 s
Saved to: C:\path\to\project\index
